# 01 — Pipeline Sentinel ETL

**Pipeline Sentinel — Computer Vision + ETL**

We start with the part that makes every later model replaceable: a clean ingest path and a canonical frame manifest.

> **Defensive training scope:** sensing, detection, tracking, anomaly scoring, and human-facing alerts for a fictional pipeline corridor. No automated engagement or weapons logic.

## Learning objectives

- Create deterministic EO/IR demo data with ground truth.
- Extract video into frame records with a reusable ingest adapter.
- Separate raw, interim, processed, and output data.
- Validate the data contract before introducing a model.

In [ ]:
from pathlib import Path
import sys
HERE=Path.cwd().resolve()
ROOT=HERE.parent if HERE.name=='notebooks' else HERE
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
print('Project root:',ROOT)

## 1. The engineering question

Before asking *which detector is best?*, ask: **Can every detector consume exactly the same data contract?**

Our canonical unit is a `FrameRecord`. The source could later be FMV, thermal video, still imagery, or a stream; downstream code should not care.

In [ ]:
from src.types import FrameRecord, Detection
from src.synthetic import generate_demo_video
from src.ingest import OpenCVVideoIngestAdapter
import pandas as pd
FrameRecord.__annotations__

## 2. Build tiny deterministic EO and IR source videos

The synthetic sequence provides known labels so later notebooks can measure errors instead of just eyeballing results.

In [ ]:
raw=ROOT/'data/raw'; raw.mkdir(parents=True,exist_ok=True)
eo_video=raw/'pipeline_demo_eo.mp4'; eo_gt=raw/'pipeline_demo_eo_gt.csv'
ir_video=raw/'pipeline_demo_ir.mp4'; ir_gt=raw/'pipeline_demo_ir_gt.csv'
if not eo_video.exists(): generate_demo_video(eo_video,eo_gt,modality='EO')
if not ir_video.exists(): generate_demo_video(ir_video,ir_gt,modality='IR')
print(eo_video,eo_video.stat().st_size,'bytes')
print(ir_video,ir_video.stat().st_size,'bytes')

In [ ]:
gt=pd.read_csv(eo_gt)
print('Ground-truth rows:',len(gt))
print('Labels:',gt.label.value_counts().to_dict())
gt.head()

### Ground truth reminder

The CSV is *not teaching DINO what these objects are*. It is our answer key. Later we compare model outputs with these annotations. Keep **training data**, **model input**, and **evaluation ground truth** conceptually separate.

## 3. Ingest with OpenCV behind an adapter

In [ ]:
frames_dir=ROOT/'data/interim/frames/eo'
ingest=OpenCVVideoIngestAdapter(sample_every_n=2)
manifest=ingest.extract(eo_video,frames_dir,sensor_id='EO_CAM_01',modality='EO')
manifest.head()

In [ ]:
manifest_path=ROOT/'data/interim/eo_manifest.parquet'
try:
    manifest.to_parquet(manifest_path,index=False)
except Exception as e:
    manifest_path=manifest_path.with_suffix('.csv'); manifest.to_csv(manifest_path,index=False); print('Parquet unavailable; wrote CSV:',e)
print('Rows:',len(manifest)); print('Saved:',manifest_path)

## 4. Inspect a frame + protected corridor

In [ ]:
import cv2, matplotlib.pyplot as plt
sample_path=Path(manifest.iloc[len(manifest)//2].image_path)
img=cv2.imread(str(sample_path))
plt.figure(figsize=(12,6)); plt.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB)); plt.title(sample_path.name); plt.axis('off');

## 5. ETL validation checks

In [ ]:
assert manifest.frame_id.is_unique
assert (manifest.width>0).all() and (manifest.height>0).all()
assert manifest.timestamp_s.is_monotonic_increasing
missing=[p for p in manifest.image_path if not Path(p).exists()]
assert not missing,missing[:3]
print('PASS — manifest is internally consistent.')

## What we accomplished

We now have a repeatable **extract → validate → manifest** path. In production this is where timestamps, sensor geometry, platform metadata, checksums, and message IDs would also be normalized.

**Next:** intentionally use a weak detector—classical motion detection—to establish a cheap baseline.